# 安装与运行示例

此 notebook 提供一个可复制的快速上手示例，演示如何：配置 `.env`、安装基础依赖、启动工具后端（FastAPI），以及如何从 Python 检查工具列表接口。

说明中的命令可在 Jupyter 中运行（使用 `!` 前缀），也可以在终端中逐条执行。

## 前提与假设

- 假设你在项目根目录 `Biomni_molecule/` 下运行此 notebook。
- 请准备好你的 LLM API key，并在 `CAi/.env` 中填写（下面的 cell 会写入一个模板文件到 `CAi/.env`）。
- 部分工具安装步骤依赖 conda 环境和额外依赖，可能需要管理员权限或较长时间。

In [ ]:
# 1. 写入 .env 模板（如果已存在请先备份）
from pathlib import Path
root = Path('.')
env_path = root / 'CAi' / '.env'
env_path.parent.mkdir(parents=True, exist_ok=True)
env_text = '''LLM_API_KEY=your_api_key_here
LLM_BASE_URL=your_llm_base_url_here
LLM_MODEL=claude-sonnet-4-5-20250929

TOOL_SERVER_HOST=0.0.0.0
TOOL_SERVER_PORT=8001
'''
env_path.write_text(env_text)
print('Wrote template .env to', env_path)

In [ ]:
# 2. 安装基础依赖（在 notebook 中执行）
# 优先安装项目本体（会读取 pyproject.toml），再补充运行工具服务常用依赖。
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install -e .
!{sys.executable} -m pip install uvicorn requests

print("\nInstalled project package and base runtime packages: uvicorn, requests.")

In [ ]:
# 3. 安装工具集合（可选、通常在 CAi/additional_tools/server 下运行）
# 注意：此脚本可能很耗时且依赖 conda；在 notebook 中运行前请确认你希望安装全部工具。
# 以下示例仅列出命令，默认只展示帮助，不会真正安装。若要安装某些工具，请在终端中运行相应命令.
import os
from pathlib import Path
install_sh = Path('CAi') / 'additional_tools' / 'server' / 'install_all.sh'
if install_sh.exists():
    print('Found', install_sh, '\nYou can run:')
    print('    bash', str(install_sh))
    print('\nOr install specific tools, e.g.:')
    print('    bash', str(install_sh), 'vina scscore toxicity')
else:
    print('install_all.sh not found at', install_sh)

In [ ]:
# 4. 启动工具后端（在后台运行示例）
# 请优先在终端运行下面的命令，Notebook 中演示如何用 nohup 在后台启动。
from pathlib import Path
app_py = Path('CAi') / 'additional_tools' / 'server' / 'app.py'
if app_py.exists():
    print('Starting tool server (background via nohup). Logs -> tool_server.log')
    # 注意：Jupyter 的子进程管理与终端不同，推荐在独立终端中运行以下命令：
    print('\nRecommended (run in terminal):')
    print('    python', str(app_py))
    print('\nNotebook background demo (may be killed when kernel restarts):')
    print('    !nohup python', str(app_py), '> tool_server.log 2>&1 &')
else:
    print('工具后端 app.py 未找到，请检查路径:', app_py)

In [ ]:
# 5. 简单检查：调用工具后端的 /tools 接口（在确保工具服务已启动后）
import time
import requests
url = 'http://127.0.0.1:8001/tools'
try:
    r = requests.get(url, timeout=5)
    print('Status', r.status_code)
    print(r.text)
except Exception as e:
    print('Unable to reach tool server at', url, '\n', e)
    print('\nIf the server is not running here, run the tool backend first (see previous cell).')

In [ ]:
import os 
from CAi.CAi_agent.agent import A1pro
from CAi.config import LLM_BASE_URL,LLM_API_KEY

agent = A1pro(
    # llm="claude-sonnet-4-5-20250929",
    # llm = "Qwen/Qwen3-8B", # free
    # llm = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B", # free
    # llm = "gemini-3-flash-preview-thinking",
    llm = "qwen3.6-plus",
    # llm = "glm-4.7-flash", # free
    # llm = "glm-5",
    # llm = "Qwen/Qwen3-Coder-30B-A3B-Instruct",
    # llm = "x-ai/grok-4-fast",
    source="Custom",
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    use_tool_retriever=False,
    expected_data_lake_files=[],
    auto_load_tools=True,  # 自动加载工具
    auto_load_skills=False,  # 不自动加载技能
)

prompt = "请介绍一下你自己，并列举你有哪些工具可以使用。"

agent.go(prompt)



## 启动 Agent UI

在项目根目录（`molecule/`）下运行：

```bash
python CAi/main.py
```

这会启动 Gradio UI（或你在 `CAi/main.py` 中配置的其它界面）。如果你只是想在后台运行，也可以在终端使用 nohup 或 tmux/screen。

如果需要指定自定义 LLM 或其它配置，请编辑 `CAi/.env` 或 `CAi/config.py` 中的相关参数。

## 快速示例：使用 curl 查看工具列表（在终端中运行）

```bash
curl -s http://127.0.0.1:8001/tools | jq
```

如果你没有安装 `jq`，可以直接用 `curl` 查看原始 JSON。

常见故障排查提示：
- 请确保 `CAi/additional_tools/server/app.py` 已在 `TOOL_SERVER_PORT` 指定的端口上运行（默认 8001）。
- 若端口被占用，使用 `lsof -i :8001` 或 `ss -ltnp` 查找占用进程并释放。
- 日志文件 `tool_server.log`（若以 nohup 启动）会包含启动错误信息。